In [7]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [8]:
word_to_ix = {"hello": 0, "world": 1}
embeds = nn.Embedding(2, 5) # 2 words in vocab, 5-dimensional embeddings
lookup_tensor = torch.tensor([word_to_ix["hello"]], dtype=torch.long)
hello_embed = embeds(lookup_tensor)
print(hello_embed)

tensor([[ 0.6614,  0.2669,  0.0617,  0.6213, -0.4519]],
       grad_fn=<EmbeddingBackward0>)


In [11]:
CONTEXT_SIZE = 2
EMBEDDING_DIM = 10
test_sentence = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.""".split()
print(test_sentence)

['When', 'forty', 'winters', 'shall', 'besiege', 'thy', 'brow,', 'And', 'dig', 'deep', 'trenches', 'in', 'thy', "beauty's", 'field,', 'Thy', "youth's", 'proud', 'livery', 'so', 'gazed', 'on', 'now,', 'Will', 'be', 'a', "totter'd", 'weed', 'of', 'small', 'worth', 'held:', 'Then', 'being', 'asked,', 'where', 'all', 'thy', 'beauty', 'lies,', 'Where', 'all', 'the', 'treasure', 'of', 'thy', 'lusty', 'days;', 'To', 'say,', 'within', 'thine', 'own', 'deep', 'sunken', 'eyes,', 'Were', 'an', 'all-eating', 'shame,', 'and', 'thriftless', 'praise.', 'How', 'much', 'more', 'praise', "deserv'd", 'thy', "beauty's", 'use,', 'If', 'thou', 'couldst', 'answer', "'This", 'fair', 'child', 'of', 'mine', 'Shall', 'sum', 'my', 'count,', 'and', 'make', 'my', 'old', "excuse,'", 'Proving', 'his', 'beauty', 'by', 'succession', 'thine!', 'This', 'were', 'to', 'be', 'new', 'made', 'when', 'thou', 'art', 'old,', 'And', 'see', 'thy', 'blood', 'warm', 'when', 'thou', "feel'st", 'it', 'cold.']


In [38]:
ngrams = [
    (
        [test_sentence[i - j - 1] for j in range(CONTEXT_SIZE)],
        test_sentence[i]
    )
    for i in range(CONTEXT_SIZE, len(test_sentence))]
print("ngrams ", ngrams[:3])

vocab = set(test_sentence)
print("vocab: ", vocab)
word_to_ix = {word : i for i, word in enumerate(vocab)}
print("word_to_ix: ", word_to_ix)

class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, contex_size):
        super(NGramLanguageModeler, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(contex_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embedding(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

ngrams  [(['forty', 'When'], 'winters'), (['winters', 'forty'], 'shall'), (['shall', 'winters'], 'besiege')]
vocab:  {'besiege', 'new', 'shame,', 'livery', "beauty's", 'field,', 'Where', 'thou', 'within', 'by', 'be', 'much', 'lies,', 'fair', 'Will', 'How', 'a', 'an', 'thriftless', 'Shall', 'his', 'dig', 'made', 'deep', 'lusty', 'thine', 'worth', 'now,', 'praise.', 'small', 'mine', 'old', 'the', 'own', 'all', 'more', 'and', 'This', "youth's", 'Proving', "deserv'd", 'Were', 'to', 'my', "feel'st", "'This", 'where', "excuse,'", 'sum', 'trenches', 'being', 'To', 'art', 'beauty', 'say,', 'When', 'it', 'succession', 'Then', 'old,', 'asked,', 'And', 'praise', 'use,', 'on', 'answer', 'blood', 'so', 'winters', 'sunken', 'days;', 'shall', 'brow,', 'couldst', 'thine!', 'were', 'gazed', 'If', 'eyes,', 'of', 'weed', 'forty', 'proud', 'treasure', 'all-eating', 'held:', 'cold.', 'when', 'count,', 'in', 'warm', 'make', 'child', "totter'd", 'Thy', 'thy', 'see'}
word_to_ix:  {'besiege': 0, 'new': 1, 'sha

In [43]:
losses = []
loss_function = nn.NLLLoss()
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for context, target in ngrams:
        #print("context: ", context)
        #print("target: ", target)
        context_idx =  torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)
        #print(context_idx)
        model.zero_grad()

        logs_probs = model(context_idx)

        loss = loss_function(logs_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    losses.append(total_loss)

losses
        

[523.9149603843689,
 521.3827157020569,
 518.8704252243042,
 516.3764672279358,
 513.899778842926,
 511.43871903419495,
 508.99289083480835,
 506.5616707801819,
 504.1450426578522,
 501.742311000824]

In [42]:
model.embedding.weight[word_to_ix["beauty"]]

tensor([ 0.4603,  1.2188, -1.9800,  0.3264, -0.0778,  0.4173,  1.1583,  0.4930,
        -0.8207, -0.7041], grad_fn=<SelectBackward0>)